# ATP PLAYERS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("silver_atp_players").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Players

In [8]:
final_tb_atp_matches = (
        tb_atp_matches 
        .withColumn(
            "PLAYER",
            f.when(
                f.col("winner_name").contains("Shevchenko"), "Aleksandr Shevchenko"
            ).otherwise(f.col("winner_name"))
        )
        .withColumn(
            "loser_name",
            f.when(
                f.col("loser_name").contains("Shevchenko"), "Aleksandr Shevchenko"
            ).otherwise(f.col("loser_name"))
        )
    )

In [9]:
window_winner_name = Window.partitionBy("winner_name")  

winners = (
        final_tb_atp_matches 
        .select(
            f.coalesce(
                f.col("winner_id"),
                f.first("winner_id", ignorenulls=True).over(window_winner_name)
            ).alias("PLAYER_ID"),
            f.col("winner_name").alias("PLAYER_NAME"),
            f.col("winner_hand").alias("PLAYER_HAND"),
            f.col("winner_ht").cast('int').alias("PLAYER_HEIGHT"),
            f.col("winner_ioc").alias("PLAYER_COUNTRY"),
            f.to_date(f.col("tourney_date").cast("string"), "yyyyMMdd").alias("MATCH_DATE"),
            f.col("winner_age").cast("double").alias("PLAYER_AGE")
        )
        .distinct()
    )

In [10]:
window_loser_name = Window.partitionBy("loser_name")  

losers = (
    final_tb_atp_matches 
    .select(
        f.coalesce(
            f.col("loser_id"),
            f.first("loser_id", ignorenulls=True).over(window_loser_name)
        ).alias("PLAYER_ID"),
        f.col("loser_name").alias("PLAYER_NAME"),
        f.col("loser_hand").alias("PLAYER_HAND"),
        f.col("loser_ht").cast("int").alias("PLAYER_HEIGHT"),
        f.col("loser_ioc").alias("PLAYER_COUNTRY"),
        f.to_date(f.col("tourney_date").cast("string"), "yyyyMMdd").alias("MATCH_DATE"),
        f.col("loser_age").cast("double").alias("PLAYER_AGE")
    )
)

In [21]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("PLAYER_ID").orderBy(f.col("MATCH_DATE").desc())

df = (
    winners.unionByName(losers)
    .withColumn(
        "PLAYER_BIRTH_DATE",
        f.expr("date_sub(MATCH_DATE, cast(PLAYER_AGE * 365.25 as int))")
    )
    .withColumn("row_num", f.row_number().over(window_spec))
    .filter(f.col("row_num") == 1)

    .groupBy("PLAYER_NAME")
    .agg(
        f.last("PLAYER_ID").alias("PLAYER_ID"),
        f.last("PLAYER_HAND").alias("PLAYER_HAND"),
        f.last("PLAYER_HEIGHT").alias("PLAYER_HEIGHT"),
        f.last("PLAYER_COUNTRY").alias("PLAYER_COUNTRY"),
        f.last("PLAYER_BIRTH_DATE").alias("PLAYER_BIRTH_DATE")
    )
)

## Save dataframe

### Local

In [22]:
df.toPandas().to_csv(
    r"../../data/silver/tb_atp_players.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)